# 02. Feature Engineering & Preprocessing

This notebook covers the feature engineering and preprocessing steps:
- Handling outliers by clipping.
- Creating calendar features (`hour`, `dayofweek`, `month`, `is_weekend`).
- Creating Fourier features (`sin_24`, `cos_24`, `sin_168`, `cos_168`).
- Creating lag features (`OT_lag_1`, `OT_lag_2`, etc.) and rolling statistics.
- Splitting into train/validation/test chronologically (70% / 15% / 15%).
- Performing Z-score normalization (fitting only on the training set to prevent leakage).
- Saving the processed splits to `data/processed/`.

In [1]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd

from src.data_loader import load_data, split_data
from src.features import handle_outliers, create_calendar_features, create_fourier_features, create_lag_features, create_rolling_features, scale_datasets

## 1. Load Raw Data

In [2]:
filepath = "../data/raw/ETTh1.csv"
df = load_data(filepath)
df.head()

,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


## 2. Handle Outliers

In [3]:
cols = [c for c in df.columns if c != 'date']
df = handle_outliers(df, cols, method='iqr')
df.head()

,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT
0,2016-07-01 00:00:00,5.827,2.009,1.599,0.462,4.203,1.340,30.531000
1,2016-07-01 01:00:00,5.693,2.076,1.492,0.426,4.142,1.371,27.787001
2,2016-07-01 02:00:00,5.157,1.741,1.279,0.355,3.777,1.218,27.787001
3,2016-07-01 03:00:00,5.090,1.942,1.279,0.391,3.807,1.279,25.044001
4,2016-07-01 04:00:00,5.358,1.942,1.492,0.462,3.868,1.279,21.948000


## 3. Create Features

In [4]:
# Calendar features
df = create_calendar_features(df, 'date')

# Fourier features (periods of 24 hours and 168 hours)
df = create_fourier_features(df, 'date', periods=[24, 168])

# Lag features for the target column 'OT'
df = create_lag_features(df, ['OT'], lags=[1, 2, 3, 24, 48, 168])

# Rolling features for target column 'OT'
df = create_rolling_features(df, ['OT'], windows=[24, 168])

# Drop rows with NaN values resulting from lags and rolling statistics
df = df.dropna().reset_index(drop=True)
print("Shape after feature engineering and dropping NaNs:", df.shape)
df.head()

Shape after feature engineering and dropping NaNs: (17252, 26)


,date,HUFL,HULL,MUFL,MULL,LUFL,LULL,OT,hour,dayofweek,...,OT_lag_1,OT_lag_2,OT_lag_3,OT_lag_24,OT_lag_48,OT_lag_168,OT_roll_mean_24,OT_roll_std_24,OT_roll_mean_168,OT_roll_std_168
0,2016-07-08 00:00:00,11.855,4.756,8.884,2.985,2.955,1.523,31.093000,0,4,...,31.586000,32.640999,33.625999,28.561001,28.913000,30.531000,30.832542,3.121859,26.836554,4.695012
1,2016-07-08 01:00:00,10.382,4.756,7.995,2.630,2.498,1.340,30.742001,1,4,...,31.093000,31.586000,32.640999,28.771999,30.389999,27.787001,30.938042,3.084314,26.839899,4.697860
2,2016-07-08 02:00:00,11.186,5.224,8.706,2.594,2.376,1.401,29.334999,2,4,...,30.742001,31.093000,31.586000,27.716999,31.514999,27.787001,31.020125,3.050188,26.857488,4.706950
3,2016-07-08 03:00:00,10.248,4.555,7.747,2.736,2.437,1.492,28.771999,3,4,...,29.334999,30.742001,31.093000,28.139000,31.233999,25.044001,31.087542,2.991319,26.866702,4.710295
4,2016-07-08 04:00:00,10.918,4.086,8.244,2.594,2.315,1.462,29.124001,4,4,...,28.771999,29.334999,30.742001,27.365000,30.531000,21.948000,31.113917,2.966882,26.888893,4.710438


## 4. Chronological Splitting

In [5]:
train_df, val_df, test_df = split_data(df, train_ratio=0.7, val_ratio=0.15)
print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")

Train shape: (12076, 26)
Val shape: (2588, 26)
Test shape: (2588, 26)


## 5. Normalize Data (Z-score)

In [6]:
# Normalizing all columns except 'date'
numeric_cols = [col for col in train_df.columns if col != 'date']

train_scaled, val_scaled, test_scaled, mean, std = scale_datasets(train_df, val_df, test_df, numeric_cols)
print("Mean values used for scaling:", mean.head(5))
print("Std values used for scaling:", std.head(5))

Mean values used for scaling: HUFL    7.871184
HULL    1.924165
MUFL    5.111472
MULL    0.677043
LUFL    2.866667
dtype: float64
Std values used for scaling: HUFL    5.177417
HULL    2.093051
MUFL    4.662198
MULL    1.916493
LUFL    1.028451
dtype: float64


## 6. Save Processed Datasets

In [7]:
os.makedirs('../data/processed', exist_ok=True)
train_scaled.to_csv('../data/processed/train.csv', index=False)
val_scaled.to_csv('../data/processed/val.csv', index=False)
test_scaled.to_csv('../data/processed/test.csv', index=False)
print("Saved scaled splits successfully.")

Saved scaled splits successfully.
